# 03c - RM-c: frozen encoder + Retrieval-Augmented Classification

RM-c tidak melatih apa pun. Ia memakai ulang dua artefak milik RM-b, yaitu
embedding beku dan head yang sudah terlatih, lalu menambahkan cabang retrieval.

Alur per sampel:

1. `p_bert` = softmax(head(embedding))
2. Cari k tetangga terdekat di indeks FAISS yang dibangun HANYA dari split train,
   lalu ubah label tetangga menjadi distribusi `p_retr`
3. `p_final = (1 - alpha) * p_bert + alpha * p_retr`, prediksi = argmax

Fusi dilakukan pada level probabilitas dan softmax hanya diterapkan sekali, di
cabang BERT sebelum fusi. Karena kedua masukan sudah berupa distribusi dan bobot
fusinya berjumlah satu, hasilnya sudah menjadi distribusi sah; softmax kedua
hanya akan meratakan selisih dan bisa mengubah argmax pada kasus nyaris seri.

Prasyarat: `03b_rmb_frozen.ipynb` sudah dijalankan (butuh `rmb_best.pt`).

In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

## 1. Head RM-b yang diwarisi

In [ ]:
head, head_config = runner._load_best_head()
print("konfigurasi head RM-b terbaik:", head_config)
print("indeks FAISS akan dibangun dari", len(runner.features.labels["train"]), "vektor train")

Indeks dibangun eksklusif dari split train. Kalau val atau test ikut masuk,
retrieval akan menemukan sampel uji di dalam indeksnya sendiri dan hasilnya
kehilangan makna.

## 2. Konfigurasi

In [ ]:
from src.models.schemas import RMCConfig

config = RMCConfig()
print(config.model_dump())

## 3. Jalankan

In [ ]:
row = runner.run(
    "rmc",
    config.model_dump(),
    note="baseline RAC alpha=0,3 k=5 mengikuti Yu dkk. 2023",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro    : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi     : {row['val_f1_judi']:.4f}")
print(f"  vektor di indeks: {row['index_vectors']:,}")
print(f"  waktu evaluasi  : {row['eval_time_s']:.2f} s")
print(f"  trainable params: {row['trainable_params']}")
print(f"  waktu latih     : {row['train_time_s']}")

## 4. Pengaruh alpha

In [ ]:
import pandas as pd

sweep = runner.run_batch(
    "rmc",
    [
        {"config": {"alpha": alpha, "k": 5},
         "note": f"sweep alpha={alpha} pada k=5"}
        for alpha in (0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0)
    ],
    batch_id="rmc_sweep_alpha",
)

sweep[["run_id", "alpha", "k", "val_f1_macro", "val_f1_judi"]]

`alpha=0` identik dengan RM-b murni dan `alpha=1` membuang cabang BERT
sepenuhnya, sehingga kedua ujung itu berfungsi sebagai pemeriksaan kewarasan:
kolom pertama harus sama persis dengan F1 RM-b.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep["alpha"], sweep["val_f1_macro"], marker="o", label="F1-macro")
ax.plot(sweep["alpha"], sweep["val_f1_judi"], marker="s", label="F1 judi")
ax.set_xlabel("alpha (bobot cabang retrieval)")
ax.set_ylabel("F1 (validation)")
ax.set_title("Pengaruh bobot fusi RAC")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Ringkasan

RM-c menambah nol trainable parameter dan nol waktu latih di atas RM-b; seluruh
biaya tambahannya ada di inferensi, yaitu pembangunan indeks sekali dan
penelusuran k tetangga per prediksi. Biaya itu diukur terpisah di
`06_analysis_export.ipynb`.

Lanjut ke `04_tuning_campaign.ipynb`.